<a href="https://colab.research.google.com/github/Sitthisak17SM/Super_AI/blob/main/hak3_600817_%E0%B8%AA%E0%B8%B4%E0%B8%97%E0%B8%98%E0%B8%B4%E0%B8%A8%E0%B8%B1%E0%B8%81%E0%B8%94%E0%B8%B4%E0%B9%8C_%E0%B8%AA%E0%B8%B5%E0%B8%AB%E0%B8%A1%E0%B8%B7%E0%B9%88%E0%B8%99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

FahMai RAG


**Sections:**
- **Section 0:** Setup & LLM Test
- **Section 1:** Dense Retrieval (MiniLM)
- **Section 2:** Sparse Retrieval (BM25)
- **Section 3:** Hybrid Retrieval (RRF)


In [ ]:
!unzip /content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

Archive:  /content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001

In [ ]:
N_QUESTIONS = 100
DATA_DIR = "/content/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

In [ ]:
KB_DIR

'/content/data/knowledge_base'

---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [ ]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

In [ ]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('ThaiLLM')

In [ ]:
def ask_llm(messages, model="OpenThaiGPT", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/openthaigpt/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2048,
        "temperature": 0.0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

LLM response (no context): <think>
ผู้ใช้ถามเกี่ยวกับความสามารถในการกันน้ำของ S3 Ultra ซึ่งเป็นเรื่องสำคัญสำหรับผู้ใช้ที่ต้องการใช้งานในสภาพแวดล้อมที่เปียกชื้น
</think>

## ความสามารถในการกันน้ำของ S3 Ultra

**S3 Ultra** มีความสามารถในการกันน้ำตามมาตรฐาน IP68

### รายละเอียดความสามารถในการกันน้ำ

| รายการ | รายละเอียด |
|--------|------------|
| **มาตรฐาน** | IP68 |
| **ความลึก** | 1.5 เมตร |
| **ระยะเวลา** | 30 นาที |
| **กันน้ำได้** | ใช่ |
| **กันฝุ่นได้** | ใช่ (IP68) |

### หมายเหตุสำคัญ

1. **ไม่ได้รับการรับรอง** สำหรับการดำน้ำหรือใช้งานในน้ำลึก
2. **ไม่ควรใช้งาน** ในสภาพแวดล้อมที่มีความดันสูง เช่น ใต้น้ำลึก
3. **ควรปิดฝา** ทุกครั้งที่ไม่ได้ใช้งาน
4. **หลีกเลี่ยงการสัมผัส** กับน้ำยาทำความสะอาดหรือสารเคมี

### คำแนะนำ

- หลีกเลี่ยงการใช้งานในสภาพแวดล้อมที่เปียกชื้นมากเกินไป
- ทำความสะอาดและเช็ดให้แห้งทุกครั้งหลังใช้งานในน้ำ
- ตรวจสอบสภาพตัวเครื่องและช่องเสียบบ่อยๆ

**S3 Ultra** ออกแบบมาเพื่อใช้งานในชีวิตประจำวันที่อาจมีความเปียกชื้นเล็กน้อย เช่น ฝนตก หรือน้ำหก แต่ไม่เหมาะสำหรับการ

### Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

Loaded 100 questions, using first 100 for this run

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Chunking


In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# 1. กำหนดโครงสร้าง Markdown ที่เราต้องการให้มันใช้เป็นจุดตัด (Split)
headers_to_split_on = [
    ("#", "Header 1"),    # เช่น ชื่อสินค้า หรือ ชื่อหมวดหลัก
    ("##", "Header 2"),   # เช่น สเปก, การรับประกัน
    ("###", "Header 3"),  # เช่น รายละเอียดขนาดย่อย
]

# สร้างตัวตัดเอกสารตามหัวข้อ Markdown
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 2. (แผนสำรอง) หากข้อมูลใต้หัวข้อนั้นๆ ยังยาวเกินไป เราจะสับย่อยอีกรอบให้ขนาดไม่เกินหน้าต่างของ LLM
chunk_size = 1000
chunk_overlap = 150
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

chunks = []

# วนลูปอ่านเอกสารทีละไฟล์
for doc in documents:
    # ตัดเอกสารรอบแรกตามหัวข้อ Markdown (จะได้ข้อความพร้อม Metadata ว่ามาจากหัวข้อไหน)
    md_header_splits = markdown_splitter.split_text(doc["text"])

    # ตัดเอกสารรอบสอง ถ้าย่อหน้านั้นยาวเกิน chunk_size
    final_splits = text_splitter.split_documents(md_header_splits)

    for split in final_splits:
        # --- การทำ Context-Enriched Chunking (เสริมบริบท) ---
        # สมมติ metadata คือ {'Header 1': 'Watch S3 Ultra', 'Header 2': 'การกันน้ำ'}
        # เราจะจับมาต่อกันเป็น "Watch S3 Ultra > การกันน้ำ"
        header_context = " > ".join(split.metadata.values()) if split.metadata else "เนื้อหาทั่วไป"

        # ✨ จุดสำคัญ: แปะ "ชื่อไฟล์อ้างอิง" และ "ลำดับหัวข้อ" ไว้ส่วนบนสุดของข้อความเสมอ!
        # วิธีนี้จะแก้ปัญหา "Chunk กำพร้า" (LLM อ่านแล้วไม่รู้ว่ากำลังพูดถึงสินค้าตัวไหน)
        enriched_text = f"ไฟล์อ้างอิง: {doc['path']}\nหัวข้อที่เกี่ยวข้อง: {header_context}\n---\n{split.page_content}"

        chunks.append({
            "text": enriched_text,
            "source": doc["path"],
            "metadata": split.metadata # เก็บ metadata ไว้ เผื่อนำไปทำ Filtering ใน RAG ต่อได้
        })

print(f"Created {len(chunks)} specialized chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(f"Metadata: {chunks[0]['metadata']}")
print("Text:\n" + chunks[0]["text"][:300])

Created 913 specialized chunks from 118 documents

--- Sample chunk ---
Source: policies/cancellation_policy.md
Metadata: {'Header 1': 'นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่'}
Text:
ไฟล์อ้างอิง: policies/cancellation_policy.md
หัวข้อที่เกี่ยวข้อง: นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่
---
**วันที่อัปเดต:** 1 มีนาคม 2569  
---


### 1.3 Embedding


In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

# 1. โหลดโมเดล BAAI/bge-m3
# แนะนำให้ระบุ device เป็น 'cuda' หากรันบน Colab ที่มี GPU เพื่อให้ทำงานเร็วขึ้น
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading BGE-M3 model on: {device}")

embed_model = SentenceTransformer("BAAI/bge-m3", device=device)
# ✨ เพิ่มใหม่
cross_encoder = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device)
print("Models loaded!")
# 2. ดึงเฉพาะส่วนข้อความ (Text) ออกมาจาก List ของ Chunks
chunk_texts = [c["text"] for c in chunks]

# 3. สร้าง Embeddings (แปลงข้อความเป็นเวกเตอร์)
print(f"Encoding {len(chunk_texts)} chunks. This might take a moment...")
chunk_embeddings = embed_model.encode(
    chunk_texts,
    batch_size=16,               # bge-m3 ตัวใหญ่กว่า แนะนำให้ลด batch_size ลงจาก 64 เหลือ 16 หรือ 32 เพื่อป้องกัน GPU RAM เต็ม
    show_progress_bar=True,
    normalize_embeddings=True    # สำคัญมากสำหรับการทำ Cosine Similarity ใน RAG
)

print(f"\nChunk embeddings shape: {chunk_embeddings.shape}")
# ผลลัพธ์ควรจะได้เป็น (n_chunks, 1024)
# สังเกตว่ามิติข้อมูลจะใหญ่กว่า MiniLM (384) ซึ่งเก็บรายละเอียดความหมายได้มากกว่า

Loading BGE-M3 model on: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Models loaded!
Encoding 913 chunks. This might take a moment...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]


Chunk embeddings shape: (913, 1024)


### 1.4 Retrieve



In [ ]:
!pip install rank-bm25 pythainlp

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from pythainlp.tokenize import word_tokenize

# ==========================================
# ส่วนที่ 1: สร้างฐานข้อมูลสำหรับค้นหาด้วยคีย์เวิร์ด (BM25 Index)
# (ขั้นตอนนี้รันแค่ครั้งเดียวตอนเตรียมข้อมูล)
# ==========================================
print("Tokenizing chunks for BM25...")
# เราต้องใช้ PyThaiNLP ตัดคำภาษาไทยใน Chunks เพื่อให้ BM25 นับคำได้แม่นยำ
tokenized_chunks = [word_tokenize(chunk["text"], engine="newmm") for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 Index created successfully!")

# ==========================================
# ส่วนที่ 2: ฟังก์ชัน Hybrid Search ด้วย RRF
# ==========================================
TOP_K = 10

def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """ค้นหาด้วย Vector และ BM25 แล้วนำมารวมคะแนนกันด้วย RRF"""

    # --- ฝั่งที่ 1: Dense Retrieval (หาด้วย Vector ความหมาย) ---
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    dense_scores = np.dot(chunk_embs, q_emb.T).flatten()
    # จัดอันดับเอกสารทั้งหมดตามคะแนน Vector
    dense_ranked_indices = np.argsort(dense_scores)[::-1]

    # --- ฝั่งที่ 2: Sparse Retrieval (หาด้วย BM25 คีย์เวิร์ด) ---
    tokenized_query = word_tokenize(query, engine="newmm")
    bm25_scores = bm25.get_scores(tokenized_query)
    # จัดอันดับเอกสารทั้งหมดตามคะแนน BM25
    sparse_ranked_indices = np.argsort(bm25_scores)[::-1]

    # --- ขั้นตอนที่ 3: รวมคะแนนด้วยสมการ RRF (Reciprocal Rank Fusion) ---
    # เอกสารไหนที่ได้อันดับดี (Rank ต้นๆ) จากทั้ง 2 ฝั่ง จะได้คะแนน RRF สูง
    rrf_scores = np.zeros(len(chunks))

    # ให้คะแนนจากฝั่ง Vector
    for rank, doc_idx in enumerate(dense_ranked_indices):
        rrf_scores[doc_idx] += 1.0 / (rrf_k + rank + 1) # +1 เพราะ rank ใน Python เริ่มจาก 0

    # ให้คะแนนจากฝั่ง BM25
    for rank, doc_idx in enumerate(sparse_ranked_indices):
        rrf_scores[doc_idx] += 1.0 / (rrf_k + rank + 1)

    # --- สรุปผล: ดึง Top K ที่คะแนนรวมสูงสุด ---
    final_top_indices = np.argsort(rrf_scores)[::-1][:k]
    final_scores = rrf_scores[final_top_indices]

    return final_top_indices, final_scores



# ==========================================
# ส่วนที่ 3: ทดสอบการค้นหา
# ==========================================
# สมมติว่าเอาคำถามข้อแรกจากไฟล์ CSV มาทดสอบ
q = questions[0]
print(f"คำถาม: {q['question']}\n")

# เรียกใช้ฟังก์ชัน hybrid_retrieve ตัวใหม่
idx, scores = hybrid_retrieve(q["question"], chunk_embeddings)

for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"อันดับ {rank} (คะแนน RRF = {s:.4f}) | ไฟล์อ้างอิง: [{chunks[i]['source']}]")
    print(f"เนื้อหา: {chunks[i]['text'][:200]}...")
    print("-" * 60)

Tokenizing chunks for BM25...
BM25 Index created successfully!
คำถาม: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

อันดับ 1 (คะแนน RRF = 0.0325) | ไฟล์อ้างอิง: [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
เนื้อหา: ไฟล์อ้างอิง: products/WK-SW-001_wongkhojon_watch_s3_ultra.md
หัวข้อที่เกี่ยวข้อง: วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
---
**Q: Watch S3 Ultra ต่างจาก W...
------------------------------------------------------------
อันดับ 2 (คะแนน RRF = 0.0320) | ไฟล์อ้างอิง: [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
เนื้อหา: ไฟล์อ้างอิง: products/WK-SW-002_wongkhojon_watch_s3_pro.md
หัวข้อที่เกี่ยวข้อง: วงโคจร Watch S3 Pro (WongKhoJon Watch S3 Pro) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
---
**Q: Watch S3 Pro มี ECG ไหมครับ? ต...
------------------------------------------------------------
อันดับ 3 (คะแนน RRF = 0.0318) | ไฟล์อ้างอิง: [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
เนื้อหา: ไฟล์อ้างอิง: products/WK-SW-001_wongkhojon_watch_s3_

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [ ]:
# ==========================================
# 1.5 Generate Answer (อัปเกรด Prompt ขั้นสุดยอด)
# ==========================================

# 1. กำหนด System Prompt ให้มี Persona ชัดเจน และมีกฎเหล็ก (System Guardrails)
SYSTEM_PROMPT = """คุณคือ AI ผู้ช่วยบริการลูกค้าของร้าน 'ฟ้าใหม่' (FahMai) ระดับมืออาชีพ
หน้าที่ของคุณคือตอบคำถามของลูกค้าโดยใช้ *ข้อมูลอ้างอิงที่กำหนดให้เท่านั้น* ห้ามใช้ความรู้ส่วนตัวตอบเด็ดขาด

กฎการตอบคำถาม:
1. อ่านและวิเคราะห์ข้อมูลอ้างอิงอย่างละเอียดเพื่อหาตัวเลือกที่ถูกต้องที่สุด
2. หากในข้อมูลอ้างอิง "ไม่มีคำตอบที่ชัดเจน" ให้เลือกตัวเลือก 'ไม่มีข้อมูลนี้ในฐานข้อมูล' เสมอ
3. หากคำถาม "ไม่เกี่ยวข้องกับร้านฟ้าใหม่หรือสินค้าในร้าน" ให้เลือกตัวเลือก 'คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า' เสมอ
4. ให้คุณอธิบายเหตุผลสั้นๆ ก่อนตอบ
และบรรทัดสุดท้าย **ต้อง**
ขั้นตอน:
   1. ระบุว่าคำถามถามเรื่องอะไร
   2. หาข้อมูลที่ตรงกันใน Context
   3. ตรวจสอบทุกตัวเลือกก่อนสรุป 'ANSWER: X' เท่านั้น (โดย X คือหมายเลขตัวเลือก 1-10)"""

def build_rag_prompt(question, choices, retrieved_chunks):
    """สร้างโครงสร้าง Prompt ให้ AI อ่านง่ายที่สุด"""

    # จัดรูปแบบข้อมูลอ้างอิงให้ชัดเจน
    context = "\n\n".join(
        f"[ข้อมูลอ้างอิงชิ้นที่ {i+1}]\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )

    # จัดรูปแบบตัวเลือก
    choices_text = "\n".join(f"ตัวเลือกที่ {k}: {v}" for k, v in choices.items())

    return f"""ข้อมูลอ้างอิง:
{context}

---
คำถามจากลูกค้า: {question}

ตัวเลือกที่มี:
{choices_text}

กรุณาวิเคราะห์ข้อมูลอ้างอิงเพื่อตอบคำถามลูกค้า (อย่าลืมสรุปคำตอบในรูปแบบ ANSWER: X ในบรรทัดสุดท้าย):"""

# ==========================================
# ทดสอบรันกับคำถามข้อที่ 1
# ==========================================
q = questions[0]

# ✨ เปลี่ยนมาใช้ hybrid_retrieve แทนของเดิมเพื่อให้หาข้อมูลเป๊ะขึ้น
idx, scores = hybrid_retrieve(q["question"], chunk_embeddings, k=10)
retrieved = [chunks[i] for i in idx]

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)

print(f"กำลังส่งคำถามไปให้ LLM คิด... (คำถาม: {q['question']})\n")

# เรียกใช้งาน LLM
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])

answer = parse_answer(raw)

print("="*50)
print(f"LLM ตอบกลับมาว่า (Raw Output):\n{raw}")
print("="*50)
print(f"ผลลัพธ์ที่สกัดได้ (Parsed answer): {answer}")

กำลังส่งคำถามไปให้ LLM คิด... (คำถาม: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ)

LLM ตอบกลับมาว่า (Raw Output):
<think>
คำถามคือ "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ" ต้องหาข้อมูลใน Context ที่เกี่ยวข้องกับความสามารถในการกันน้ำของ Watch S3 Ultra

ข้อมูลอ้างอิงชิ้นที่ 3 ระบุว่า "มาตรฐานกันน้ำระดับ 100 เมตร (10 ATM)" ซึ่งตรงกับคำถามที่ถามถึง ATM ดังนั้นคำตอบที่ถูกต้องคือ 10 ATM ซึ่งตรงกับตัวเลือกที่ 5
</think>

คำถามนี้ถามถึงความสามารถในการกันน้ำของ Watch S3 Ultra โดยเฉพาะอย่างยิ่งในหน่วย ATM (Atmosphere) ซึ่งเป็นหน่วยที่ใช้วัดความดัน

จากข้อมูลอ้างอิงชิ้นที่ 3 ซึ่งเป็นรายละเอียดสินค้าของ Watch S3 Ultra ระบุชัดเจนว่า "มาตรฐานกันน้ำระดับ 100 เมตร (10 ATM)" ซึ่งหมายความว่านาฬิกานี้สามารถทนต่อความดันน้ำได้ถึง 10 ATM หรือเทียบเท่ากับความลึก 100 เมตร

ดังนั้น คำตอบที่ถูกต้องคือ 10 ATM ซึ่งตรงกับตัวเลือกที่ 5

ANSWER: 5
ผลลัพธ์ที่สกัดได้ (Parsed answer): 5


In [ ]:
retrieved

[{'text': 'ไฟล์อ้างอิง: products/WK-SW-001_wongkhojon_watch_s3_ultra.md\nหัวข้อที่เกี่ยวข้อง: วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้\n---\n**Q: Watch S3 Ultra ต่างจาก Watch S3 Pro ยังไงบ้างครับ?**\nA: S3 Ultra แตกต่างจาก S3 Pro ในหลายด้านครับ ได้แก่ (1) ตัวเรือนไทเทเนียม เบากว่าสแตนเลสของ S3 Pro, (2) กันน้ำ 100 เมตร พร้อม Dive Mode — S3 Pro กันน้ำ 50 เมตรแต่ไม่มี Dive Mode, (3) GPS แบบ Dual-Band แม่นยำกว่า S3 Pro ที่เป็น Single-Band, (4) หน้าจอใหญ่กว่า 1.9 นิ้ว vs 1.7 นิ้ว, (5) แบตเตอรี่อึดกว่า ทั้งสองรุ่นมี ECG และ NFC Pay เหมือนกัน  \n**Q: Dive Mode ใช้ดำน้ำได้ลึกแค่ไหนครับ?**\nA: Dive Mode ของ S3 Ultra รองรับการดำน้ำสูงสุด 40 เมตร (ลึกกว่านั้นนาฬิกาจะไม่รับประกันความถูกต้องของข้อมูลและอาจเกิดความเสียหายได้) ในระหว่างดำน้ำจะแสดงผลความลึก อุณหภูมิน้ำ และเวลาดำน้ำ แต่ขอแนะนำว่าไม่ควรดำน้ำในน้ำร้อน น้ำพุร้อน หรือน้ำทะเลที่มีแรงดัน เพราะอาจส่งผลต่อซีลกันน้ำได้  \n**Q: ECG ใช้งานได้ไหม ถ้าใช้ Android ทั่วไป (ไม่ใช่ FahMai)?**\nA: ได้ครับ ฟังก์

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [ ]:
import time
import pandas as pd

# ── 1. Query Expansion ──────────────────────────────────────
def expand_query(question):
    resp = ask_llm([{"role": "user", "content":
        f"จงเขียนคำค้นหา 3 แบบที่ต่างกันสำหรับคำถามนี้ คั่นด้วย | : {question}"}])
    time.sleep(0.3)
    if resp is None:
        return [question]
    return [question] + resp.split("|")[:2]


# ── 2. Rerank ────────────────────────────────────────────────
def rerank(query, candidate_chunks, top_n=5):
    pairs = [(query, c["text"][:256]) for c in candidate_chunks]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, candidate_chunks), reverse=True)
    return [chunk for _, chunk in ranked[:top_n]]


# ── 3. Hybrid Retrieve Chunks (รวมทุกอย่าง) ─────────────────
def hybrid_retrieve_chunks(query):
    queries = expand_query(query)

    combined_scores = np.zeros(len(chunks))
    for q in queries:
        idx, scores = hybrid_retrieve(q, chunk_embeddings, k=20)
        for i, s in zip(idx, scores):
            combined_scores[i] += s

    # RRF ดึง 20 candidates
    top_idx = np.argsort(combined_scores)[::-1][:20]
    candidates = [chunks[i] for i in top_idx]

    # Cross-encoder คัดเหลือ 5
    return rerank(query, candidates, top_n=5)

# 2. ฟังก์ชันหลักสำหรับรัน Pipeline
def run_pipeline(questions_list, retrieve_fn, label="hybrid"):
    """Run retrieval + LLM on all questions. Returns predictions dict."""
    predictions = {}

    print(f"🚀 เริ่มรันโมเดลด้วยระบบ: {label} ...")

    for i, q in enumerate(questions_list):
        # 1. ค้นหาเอกสารด้วย Hybrid Search
        retrieved_chunks = retrieve_fn(q["question"])

        # 2. ประกอบร่าง Prompt
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)

        # 3. ส่งให้ ThaiLLM คิดและตอบ
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ])

        # 4. สกัดคำตอบ
        pred = parse_answer(raw)

        # Fallback: ถ้า AI รวนจนตอบไม่ตรงฟอร์แมต ให้เดาข้อ 9 (ไม่มีข้อมูล) หรือข้อ 1 ไปก่อน
        # ในที่นี้แนะนำให้เดาข้อ 9 ปลอดภัยกว่าการเดาข้อ 1 เพราะถ้าหาไม่เจอมักจะแปลว่าไม่มีข้อมูล
        if pred is None:
            pred = 9
            print(f"  ⚠️ Q{q['id']:>3}: Parse failed, using fallback (9)")

        predictions[q["id"]] = pred

        print(f"  ✅ Q{q['id']:>3}: pred={predictions[q['id']]}")

        # พักเบรกเล็กน้อย ป้องกัน Rate Limit จาก API
        time.sleep(0.3)

    print(f"\n🎉 {label}: ตอบคำถามเสร็จสิ้นทั้งหมด {len(predictions)} ข้อ")
    return predictions

# ==========================================
# 3. เริ่มรันจริงกับคำถามทั้งหมด (สมมติว่าตัวแปร questions มีครบ 100 ข้อ)
# ==========================================
# รันเลย!
hybrid_preds = run_pipeline(questions, hybrid_retrieve_chunks, label="Hybrid-RAG")

# ==========================================
# 4. เซฟผลลัพธ์เป็นไฟล์ submission.csv เพื่อส่ง Kaggle / ระบบตรวจ
# ==========================================
submission_df = pd.DataFrame({
    "id": list(hybrid_preds.keys()),
    "answer": list(hybrid_preds.values())
})

# บันทึกเป็นไฟล์ CSV
submission_df.to_csv("submission.csv", index=False)
print("\n💾 บันทึกไฟล์ submission.csv เรียบร้อยแล้ว!")